# SpreadsheetBench Verified-400: MiniMax M3 + fabric-rlm

Runs the full SpreadsheetBench Verified-400 with MiniMax M3 through OpenRouter and grades saved target cells against golden workbooks, writing JSONL/summary artifacts. Works in a Fabric notebook (attach a Lakehouse first) or on a plain machine. Our verified run on fabric-rlm 0.2.8, with the excel_modify skill and workbook structure context, scored 330 of 400 (82.5 percent) under the benchmark's own evaluation.py run unmodified, for $2.61 of model spend.

Do not edit the benchmark set or inspect golden answers during execution. The sanity validator below checks only invalid artifact states such as formulas, Excel errors, placeholders, or code/prose in target cells; it does not compare to golden workbooks.

In [ ]:
# Parameters
GIT_COMMIT = "v0.5.0"
MODEL = "openrouter/minimax/minimax-m3"
RUN_ID = "ssb400-minimax-m3-frozen"
LIMIT = 400
MAX_TURNS = 14
TIMEOUT = 300
MAX_TOKENS = 16000
TEMPERATURE = 1.0
HF_URL = "https://huggingface.co/datasets/KAKA22/SpreadsheetBench/resolve/main/spreadsheetbench_verified_400.tar.gz"

In [ ]:
%pip install -q openpyxl dspy-ai litellm "fabric-rlm==0.5.0"

In [ ]:
import getpass, os
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

In [ ]:
import hashlib, json, pathlib, re, shutil, tarfile, time, traceback, urllib.request
from typing import Any

import dspy, openpyxl
from fabric_rlm import File, RLM, add_excel_workbook_context
from fabric_rlm.skill_loader import SkillLoader

BASE = pathlib.Path("/lakehouse/default/Files/spreadsheetbench_minimax_m3") if pathlib.Path("/lakehouse/default/Files").is_dir() else pathlib.Path("spreadsheetbench_minimax_m3")
DATA = BASE / "data"
RUN = BASE / "runs" / RUN_ID
WORK = RUN / "work"
TRACES = RUN / "traces"
SUBMITTED = RUN / "submitted_xlsx"
for p in (DATA, WORK, TRACES, SUBMITTED): p.mkdir(parents=True, exist_ok=True)

def dataset_dir():
    marker = DATA / "spreadsheetbench_verified_400"
    if (marker / "dataset.json").exists(): return marker
    raw = DATA / "spreadsheetbench_verified_400.tar.gz"
    if not raw.exists(): urllib.request.urlretrieve(HF_URL, raw)
    ex = DATA / "_extract"
    if ex.exists(): shutil.rmtree(ex)
    ex.mkdir(parents=True)
    with tarfile.open(raw) as tf: tf.extractall(ex, filter='data')
    src = next(p.parent for p in ex.rglob("dataset.json"))
    if marker.exists(): shutil.rmtree(marker)
    shutil.move(str(src), str(marker))
    return marker

def rows(ds):
    out = []
    for r in json.loads((ds / "dataset.json").read_text(encoding="utf-8")):
        sid = str(r["id"])
        out.append({**r, "question_id": f"SSB_{sid}", "spreadsheet_id": sid, "init_file": f"1_{sid}_init.xlsx", "golden_file": f"1_{sid}_golden.xlsx"})
    return out[:LIMIT]

def parse_pos(pos):
    s = re.sub(r"!'", "'!", pos.strip())
    # An unmatched TRAILING quote must be dropped, not balanced by prepending
    # one: prepending flips the quote state so every later comma reads as
    # quoted and a multi-range position never splits. Without this,
    # "'Sheet1!'A1:A50,'Sheet2!'A1:E20,'Sheet3!'A1:A50'" collapses into one
    # unparseable range and openpyxl raises, failing SSB_170-13 outright.
    # A LEADING imbalance still gets a prepended quote (e.g. SSB_41-47), and a
    # sheet whose name contains commas stays intact (SSB_130-9).
    if s.count("'") % 2 and s.endswith("'"): s = s[:-1]
    elif s.count("'") % 2: s = "'" + s
    parts, cur, q = [], "", False
    for ch in s:
        if ch == "'": q = not q; continue
        if ch == "," and not q:
            if cur.strip(): parts.append(cur.strip())
            cur = ""
        else: cur += ch
    if cur.strip(): parts.append(cur.strip())
    out = []
    for p in parts:
        sh, rng = (p.split("!", 1) if "!" in p else (None, p))
        m = re.match(r"^([A-Z]+)(\d+):(\d+)$", rng.strip())
        out.append((sh.strip() if sh else None, f"{m.group(1)}{m.group(2)}:{m.group(1)}{m.group(3)}" if m else rng.strip()))
    return out

def coord_values(rng):
    if hasattr(rng, "value"): return [(rng.coordinate, rng.value)]
    return [(c.coordinate, c.value) for row in rng for c in (row if not hasattr(row, "value") else [row])]

def flat(rng): return [v for _, v in coord_values(rng)]
def eq(a, b): return (a is None and b is None) or (isinstance(a, (int,float)) and isinstance(b, (int,float)) and abs(float(a)-float(b)) <= 1e-6) or str(a).strip() == str(b).strip()

def grade(out_xlsx, gold_xlsx, sheet, pos):
    wa, wg = openpyxl.load_workbook(out_xlsx, data_only=True), openpyxl.load_workbook(gold_xlsx, data_only=True)
    hit = total = 0
    for sh, rng in parse_pos(pos):
        s = sh or sheet or wa.sheetnames[0]
        sa = s if s in wa.sheetnames else wa.sheetnames[0]
        sg = s if s in wg.sheetnames else wg.sheetnames[0]
        av, gv = flat(wa[sa][rng]), flat(wg[sg][rng])
        hit += sum(eq(a, g) for a, g in zip(av, gv)); total += len(gv)
    return hit == total, hit, total

def task_text(r, path):
    sheet = r.get("answer_sheet") or "(use the only sheet in the workbook)"
    return f"""You must MODIFY an Excel (.xlsx) workbook in place using openpyxl.
WORKBOOK PATH: {path}
TARGET SHEET: {sheet}
TARGET CELL RANGE: {r['answer_position']}
INSTRUCTION:\n{r['instruction']}
REQUIRED: inspect the workbook, compute values in Python, write literal values into exactly the target cells/ranges, save the same path, reload with data_only=True, verify no formulas/errors/placeholders/prose/code remain in target cells unless a blank is the intended output, then SUBMIT(answer='done')."""

def sanity(r, path):
    errors = {"#N/A", "#VALUE!", "#REF!", "#DIV/0!", "#NAME?", "#NULL!", "#NUM!"}
    bad = ("Sub ", "End Sub", "Power Query", "VBA", "Macro:", "let Source", "Application.", "ws.Range", "ws.Rows")
    def v(payload, context):
        assert payload.get("answer") == "done", "answer must be done"
        wf, wv = openpyxl.load_workbook(path, data_only=False), openpyxl.load_workbook(path, data_only=True)
        for sh, rng in parse_pos(r["answer_position"]):
            s = sh or r.get("answer_sheet") or wf.sheetnames[0]
            s = s if s in wf.sheetnames else wf.sheetnames[0]
            for (coord, fv), (_, dv) in zip(coord_values(wf[s][rng]), coord_values(wv[s][rng])):
                assert fv not in errors and dv not in errors, f"{s}!{coord} has Excel error"
                if isinstance(fv, str):
                    assert not fv.startswith("="), f"{s}!{coord} still contains formula"
                    assert fv not in ("-", "TBD", "N/A", "see notes"), f"{s}!{coord} has placeholder"
                    for marker in bad: assert marker not in fv, f"{s}!{coord} contains code/prose marker {marker!r}"
    return v

In [ ]:
ds = dataset_dir(); spr = ds / "spreadsheet"; all_rows = rows(ds)
lm = dspy.LM(MODEL, api_key=os.environ["OPENROUTER_API_KEY"], api_base="https://openrouter.ai/api/v1", max_tokens=MAX_TOKENS, temperature=TEMPERATURE, extra_body={"usage": {"include": True}, "provider": {"order": ["minimax/fp8"], "allow_fallbacks": False}})
skill_loader = SkillLoader(); results = []; passed_n = 0; cost_total = 0.0; t0 = time.time()
skill_path = pathlib.Path(__import__('fabric_rlm').__file__).parent / 'skills' / 'excel_modify.md'
skill_sha = hashlib.sha256(skill_path.read_bytes()).hexdigest()

for i, r in enumerate(all_rows, 1):
    qid, sid = r['question_id'], r['spreadsheet_id']
    src = spr / sid / r['init_file']; gold = spr / sid / r['golden_file']
    wdir = WORK / qid; wdir.mkdir(parents=True, exist_ok=True); work = wdir / 'work.xlsx'; shutil.copyfile(src, work)
    since = len(getattr(lm, 'history', [])); rec = {'question_id': qid, 'spreadsheet_id': sid, 'answer_sheet': r.get('answer_sheet'), 'answer_position': r['answer_position'], 'model': MODEL, 'git_commit': GIT_COMMIT, 'skill_sha256': skill_sha}
    try:
        rlm = RLM.from_task(task=add_excel_workbook_context(task_text(r, work), str(work), target_position=r['answer_position'], default_sheet=r.get('answer_sheet') or None), inputs={'workbook': File(str(work))}, outputs=['answer'], lm=lm, skill_loader=skill_loader, skills=['excel_modify'], max_turns=MAX_TURNS, timeout=TIMEOUT, output_validator_context=sanity(r, work))
        q0 = time.perf_counter(); out = rlm.run(); elapsed = time.perf_counter() - q0
        ok, hit, total = grade(work, gold, r.get('answer_sheet') or '', r['answer_position'])
        hist = getattr(lm, 'history', [])[since:]; cost = sum(float((h.get('usage') or {}).get('cost') or h.get('cost') or 0) for h in hist if isinstance(h, dict))
        cost_total += cost; passed_n += int(ok); shutil.copyfile(work, SUBMITTED / f'{qid}.xlsx')
        turns = list(getattr(out.trajectory, 'turns', []) or []) if out.trajectory else []
        (TRACES / f'trace_{qid}.json').write_text(json.dumps({'qid': qid, 'submitted': out.submitted, 'payload': out.payload, 'failure_reason': out.failure_reason, 'turns': [t.to_dict() if hasattr(t, 'to_dict') else t for t in turns]}, default=str, indent=2))
        rec.update({'passed': ok, 'cells_matched': hit, 'cells_total': total, 'submitted': out.submitted, 'failure_reason': out.failure_reason, 'elapsed_seconds': round(elapsed, 2), 'n_turns': len(turns), 'cost_usd': cost})
    except Exception as exc:
        rec.update({'passed': False, 'error': f'{type(exc).__name__}: {exc}', 'traceback': traceback.format_exc()[:2000]})
    results.append(rec)
    with (RUN / 'results_F.jsonl').open('a', encoding='utf-8') as f: f.write(json.dumps(rec, default=str) + '\n')
    print(f"[{i}/{len(all_rows)}] {qid} pass={rec.get('passed')} cells={rec.get('cells_matched',0)}/{rec.get('cells_total',0)} turns={rec.get('n_turns')} cost=${float(rec.get('cost_usd') or 0):.4f}")

summary = {'strategy': 'F', 'model': MODEL, 'skill': 'excel_modify', 'skill_sha256': skill_sha, 'source': 'verified400', 'git_commit': GIT_COMMIT, 'n': len(all_rows), 'n_passed': passed_n, 'pass_rate': passed_n / max(1, len(all_rows)), 'cost_usd': cost_total, 'total_seconds': round(time.time() - t0, 1), 'run_dir': str(RUN)}
(RUN / 'summary_F.json').write_text(json.dumps(summary, indent=2))
summary